Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.
SPDX-License-Identifier: MIT-0

# Cleanup

Delete what this module created: the online evaluation configuration, then the local state file. The
domain, the space, the Dataset integration, the access grants, the roles and Transaction Search belong to
`workshop-agentcore-stack` (the instance-role grant to `code-editor`) and are removed with those stacks —
with the account at the end of a Workshop Studio event, or with `./deploy-cfn.sh destroy` in your own
account. The participant policy has no `DeleteDomain`, `DeleteSpace` or `DeleteDatasetIntegration`.

At an AWS event, Workshop Studio recycles the account when the event ends, so this notebook is optional
there. In your own account, run it: a surviving online evaluation configuration keeps scoring and keeps
costing.

In [ ]:
%pip install --quiet "boto3==1.43.126" "botocore==1.43.126"
import boto3

assert tuple(int(x) for x in boto3.__version__.split(".")[:3]) >= (1, 43, 100), (
    f"boto3 too old: {boto3.__version__} (need >=1.43.100 for the cloudwatchomni client)"
)
# If the kernel imported an older botocore before the pip install above, the new
# service models are not loaded yet. Fail here with an actionable message.
services = boto3.session.Session().get_available_services()
assert "cloudwatchomni" in services and "observabilityadmin" in services, (
    "This kernel has no cloudwatchomni client. Restart the kernel (Kernel > Restart) and run this cell again."
)
print(f"boto3 {boto3.__version__} OK - cloudwatchomni and observabilityadmin clients available")

In [ ]:
import json
import os
import pathlib
import time

import boto3

REGION = (
    os.environ.get("AWS_REGION")
    or os.environ.get("AWS_DEFAULT_REGION")
    or boto3.session.Session().region_name
)
if not REGION:
    raise RuntimeError("No AWS region configured. Set AWS_REGION or run `aws configure set region <region>`.")
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

# The CLI pages save identifiers as `export KEY=VALUE` lines in this file, so a
# participant can switch between the CLI and notebook paths at any step.
STATE_FILE = pathlib.Path.home() / ".workshop-omni.env"


def load_state():
    state = {}
    if STATE_FILE.exists():
        for line in STATE_FILE.read_text().splitlines():
            if line.startswith("export ") and "=" in line:
                key, value = line[len("export "):].split("=", 1)
                state[key] = value.strip()
    return state


def save_state(**values):
    state = load_state()
    state.update({k: v for k, v in values.items() if v})
    STATE_FILE.write_text("".join(f"export {k}={v}\n" for k, v in state.items()))
    return state


omni = boto3.client("cloudwatchomni", region_name=REGION)
obs_admin = boto3.client("observabilityadmin", region_name=REGION)
logs = boto3.client("logs", region_name=REGION)
cfn = boto3.client("cloudformation", region_name=REGION)
print(f"Region {REGION}, account ...{ACCOUNT_ID[-4:]}")

## Step 1: Delete the online evaluation configuration

In [ ]:
control = boto3.client("bedrock-agentcore-control", region_name=REGION)
for c in control.list_online_evaluation_configs()["onlineEvaluationConfigs"]:
    if c["onlineEvaluationConfigName"] != "workshop_travel_agent":
        continue
    try:
        control.delete_online_evaluation_config(onlineEvaluationConfigId=c["onlineEvaluationConfigId"])
        print(f"Deleted online evaluation config {c['onlineEvaluationConfigId']}")
    except Exception as exc:  # report, never hide: a surviving config keeps scoring and keeps costing
        print(f"ERROR: {c['onlineEvaluationConfigId']} was NOT deleted: {exc}")

## Step 2: Remove the local state

Nothing else to delete from here: the domain, space and Dataset integration are CloudFormation's.

In [ ]:
if STATE_FILE.exists():
    STATE_FILE.unlink()
    print(f"Removed {STATE_FILE}")
else:
    print(f"{STATE_FILE} was already gone")

## Step 3: Verify

In [ ]:
print("Eval configs:", len(control.list_online_evaluation_configs()["onlineEvaluationConfigs"]))
print("Domains:     ", len(omni.list_domains().get("items", [])), "(stays 1 until the stack is deleted)")

The evaluation count is `0` (the delete completes a few seconds after the call returns, so re-run the cell above if it still shows `1`); the domain count stays `1` until `workshop-agentcore-stack` is deleted.

| Left behind | Why | Action |
|-------------|-----|--------|
| `AWSServiceRoleForCloudWatch_Omni` | Created by the service on first use; recreated when needed | Leave it |
| Domain, space, Dataset integration, access grants, Transaction Search and the three roles | Owned by `workshop-agentcore-stack` (instance-role grant: `code-editor`) | Removed with the stacks (self-paced: `./deploy-cfn.sh destroy`) |
| Spans and logs in `aws/spans` and the runtime log groups | Retention is set on the log groups | Expire on their own, or delete the groups |
| `/aws/bedrock-agentcore/evaluations/results/<id>` | Kept as a record | Delete from the CloudWatch console if you want the account clean |

Continue to the workshop cleanup page to remove the rest of the platform.